# KV-Cache Graph Prefetching — Runner Notebook

This notebook clones the repo (if not already present), installs dependencies, and runs the full pipeline:

1. **Baseline** (`kv_cache_experiment_baseline.py`) — cold vs cosine/graph prefetch
2. **Adaptive weight extraction** (`adaptive_grid_search.py`) — learns (alpha, beta) per question type from the baseline results
3. **Adaptive run** (`kv_cache_experiment_adaptive.py`) — re-runs using the learned weights
4. **API-based evaluation** (Gemini / OpenAI / Groq / OpenRouter) — separate section at the end, no GPU needed

Works both on Google Colab and on a local machine with a Python + (for steps 1-3) CUDA GPU.

**Before running:** fill in `REPO_URL` in the first code cell below with your actual GitHub repo URL.

In [ ]:
# ---- Configuration: EDIT THESE ----
REPO_URL   = "https://github.com/Fasih20/Graph-Adaptive-KV-Research.git"   # <-- TODO: put your repo URL here
REPO_DIR   = "kv-cache-graph-prefetching"   # local folder name the repo will be cloned into

MODEL_REPO = "meta-llama/Llama-3.2-3B-Instruct"   # HF model id to run through baseline + adaptive
MODEL_TAG  = "Llama3.2-3B"                        # short tag used in output filenames

DATASETS = ["hotpot", "multifield", "musique", "2wiki"]   # datasets to evaluate on
TOP_M = 20
GRAPH_CONSTRUCTION = "topm"   # "topm" or "threshold"
TORCH_DTYPE = "float16"       # "auto" / "float16" / "bfloat16"


## 0. Clone the repo (skipped if it already exists) and install dependencies

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    print(f"'{REPO_DIR}' not found locally -- cloning from {REPO_URL} ...")
    !git clone "{REPO_URL}" "{REPO_DIR}"
else:
    print(f"'{REPO_DIR}' already exists locally -- skipping clone.")

%cd {REPO_DIR}


In [ ]:
!pip install -q torch transformers huggingface_hub sentence-transformers \
    langchain-text-splitters scikit-learn scipy numpy pandas requests


## 1. Baseline run

Cold vs. cosine/threshold vs. top-m graph prefetch, no learned weights. This produces the CSVs that Step 2 (adaptive grid search) needs as input.

In [ ]:
%cd src

!python kv_cache_experiment_baseline.py \
    --model "{MODEL_REPO}" \
    --datasets {" ".join(DATASETS)} \
    --graph_construction {GRAPH_CONSTRUCTION} \
    --top_m {TOP_M} \
    --output_dir ./results \
    --torch_dtype {TORCH_DTYPE}


## 2. Adaptive weight extraction

Grid-searches `(alpha, beta)` per question type (`semantic`, `structural`, `multi-hop`) against the baseline CSV produced above, and writes `adaptive_weights_<MODEL_TAG>_<dataset>.json` for each dataset into `./adaptive_results`.

In [ ]:
for ds in DATASETS:
    csv_file = f"./results/results_{MODEL_REPO.replace('/', '_').replace('.', '_')}_{ds}_hitaware_topm{TOP_M}.csv"
    print(f"--- Grid search for dataset: {ds} ---")
    !python adaptive_grid_search.py \
        --csv_dir results \
        --csv_file "{csv_file}" \
        --dataset_key "{ds}" \
        --model_tag "{MODEL_TAG}" \
        --model_repo "{MODEL_REPO}" \
        --graph_construction {GRAPH_CONSTRUCTION} \
        --top_m {TOP_M}


> **Note:** the CSV filename pattern above matches how the baseline script names its outputs (`results_<model_with_underscores>_<dataset>_hitaware_topm<N>.csv`). If your baseline output uses a different exact filename, adjust `csv_file` accordingly, or just point `--csv_file` at the file directly.

## 3. Adaptive run

Re-runs the experiment using the learned weights from Step 2.

In [ ]:
for ds in DATASETS:
    weights_file = f"./adaptive_results/adaptive_weights_{MODEL_TAG}_{ds}.json"
    print(f"--- Adaptive run for dataset: {ds} ---")
    !python kv_cache_experiment_adaptive.py \
        --model "{MODEL_REPO}" \
        --datasets "{ds}" \
        --graph_construction {GRAPH_CONSTRUCTION} \
        --top_m {TOP_M} \
        --adaptive_weights "{weights_file}" \
        --output_dir ./results_v4_adaptive \
        --torch_dtype {TORCH_DTYPE}

print("All datasets completed!")


---

## 4. API-based evaluation (Gemini / OpenAI / Groq / OpenRouter)

This section is independent of Steps 1-3 above — no GPU required, just an API key for whichever provider you want to test.

Set **one or more** of the keys below (leave the rest as empty strings / don't run those cells).

In [ ]:
import os

# ---- Fill in whichever provider(s) you're using ----
os.environ["GEMINI_API_KEY"]     = ""   # for --provider gemini
os.environ["OPENAI_API_KEY"]     = ""   # for --provider openai --backend openai
os.environ["GROQ_API_KEY"]       = ""   # for --provider openai --backend groq
os.environ["OPENROUTER_API_KEY"] = ""   # for --provider openai --backend openrouter
os.environ["ANTHROPIC_API_KEY"]  = ""   # for --provider anthropic


In [ ]:
%cd ../src/api_eval


### 4a. Single config quick test

In [ ]:
!python qa_quality_eval.py --provider gemini --dataset hotpot --top_m 12 --K 12


### 4b. Full top-m sweep (this is what produced the `Results/API_ver` numbers)

- `--cheap` (default): 8 questions per top_m value (~24 calls) — safe for free tiers.
- `--full`: 60 questions per top_m value (~180 calls) — check your provider's quota first.
- Resumable: if a run stops on a quota error, just re-run the exact same command later — it picks up where it left off.

In [ ]:
# Gemini
!python top_m_sweep.py --provider gemini --model gemini-3.5-flash-lite --full


In [ ]:
# OpenAI-compatible backend via Groq
!python top_m_sweep.py --provider openai --backend groq --full --top_m_values 12,20


### 4c. Determinism check (optional)

Repeats the same config `--n_reps` times to check how consistent a provider's answers are across repeated calls.

In [ ]:
!python determinism_check.py --provider gemini --n_reps 2


---
## Notes for whoever is running this

- If cloning fails because the repo is private, either make it public/temporarily accessible, or clone manually beforehand and just re-run this notebook — Step 0 will detect the existing folder and skip the clone.
- Steps 1-3 need a CUDA GPU for anything beyond the smallest models (1B-3B) in reasonable time; on Colab, make sure the runtime type is set to GPU.
- Step 4 needs no GPU at all and can run on any machine, including a plain laptop.
